# 5つのモデルについて、labotestでregression

In [ ]:
from matplotlib.lines import Line2D

In [ ]:
import statsmodels.api as sm

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

import sys
from decimal import Decimal, ROUND_HALF_UP
import re
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import random
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats import multitest as multi
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error

#!conda install -c conda-forge -y watermark
%load_ext watermark
%watermark -a 'Akari FUJIMOTO' -udnt -vmw -iv

# import

## demographics

In [ ]:

fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge-CommCohort67K-ver1_'
fileName = 'demographicsDF.tsv'

tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')


tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))
demoDF=tempDF

## laboratory test : testDF

In [ ]:
#Import rename labodata

fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'laboDF-MF.tsv'

tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')


tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))
testDF=tempDF

## metabolome

In [ ]:
#import metage
cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/met/LASSO/'
    ipynbName = '261016_TMM-BAge_MetAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF

metDF = tempD

## specific health catch up

In [ ]:
#import antage

cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_AntAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF

antDF=tempD

In [ ]:
#import hemage

cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_HemAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF

hemDF=tempD

In [ ]:
#import chemage

cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_ChemAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF

chemDF=tempD

In [ ]:
#import laboage

cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_LabAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF

labDF=tempD

## laboratory model

In [ ]:
#import algage

cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/labo/LASSO/'
    ipynbName = '261016_TMM-BAge_AlgAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF

algDF=tempD

In [ ]:
#import kidage
cohortD = ['SexPooled', 'Female', 'Male']

tempD={}
for cohort in cohortD:

    fileDir = './ExportData/Savemodel/labo/LASSO/'
    ipynbName = '261016_TMM-BAge_KidAge-LASSO_'
    fileName = 'predictions-'+cohort+'.tsv'
    tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

    tempDF = tempDF.set_index('ParticipantID')
    tempD[cohort] = tempDF

#merge: female + male = sexstratified

cohortD = {'SexPooled': tempD['SexPooled'],
          'SexStratified': pd.concat([tempD['Female'],tempD['Male']], axis=0)}

#check
for cohort, tempDF in cohortD.items():
    print(cohort)
    display(tempDF)
    display(tempDF.describe(include='all'))
    tempD[cohort] = tempDF


kidDF=tempD

# 2. 変数設定

## -1 labo

検体検査情報は、血液・尿の血液学的検査、生化学的検査、免疫学的検査、尿検査
https://www.megabank.tohoku.ac.jp/researchers/biobankcohort

- hematology
- clinical_test
- allergy
- kidney
- administrative

In [ ]:
#推定される対応
test_dict = {
    'hematology': [
        '桿状核球 [%]',
        '分葉核球 [%]',
        '赤芽球 [/100WBC]',
        'A-Lymphocyte [%]',
        'Pro-Myelo [%]',
        'Myeloblast [%]',
        'Myelo [%]',
        'Meta-Myelo [%]'],

    'clinical_test': [
        'NT-proBNP',
        'グリコアルブミン [%]',
        'ペプシノーゲンⅠ [ng/mL]',
        'ペプシノーゲンⅡ [ng/mL]',
        'PG1/PG2 比 ',
        '高感度心筋トロポニンT [ng/mL]'],

    'allergy': [
        # アレルゲン関連項目
        'オオアワガエリ [LC]',
        'ハルガヤ [LC]',
        'カモガヤ [LC]',
        'カモガヤ [UA/mL]', 'ブタクサ混合物Ⅰ [LC]', 'ヨモギ [LC]', 'ヨモギ [UA/mL] ',
        'スギ [LC]', 'スギ [UA/mL] ', 'スギ [UA/mL] 岩手', 'ヒノキ [LC]',
        'ハンノキ [LC]', 'シラカンバ [LC]', 'コナヒョウヒダニ [LC]', 
        'コナヒョウヒダニ [UA/mL] ', 'コナヒョウヒダニ [UA/mL] 岩手', 'ハウスダストⅠ [LC]', 
        'ペニシリウム [LC]', 'クラドスポリウム [LC]', 'カンジダ [LC]', 'カンジダ [UA/mL]', 
        'アルテルナリア [LC]', 'アスペルギルス [LC]', 'ネコ皮屑 [LC]', 'ネコ皮屑 [UA/mL]', 
        'イヌ皮屑 [LC]', '小麦 [LC]', '大豆 [LC]', '米 [LC]', 'マグロ [LC]', 'サケ [LC]', 
        'エビ [LC]', 'エビ [UA/mL]', 'カニ [LC]', 'チェダーチーズ [LC]', 'ミルク [LC]', 
        '牛肉 [LC]', '鶏肉 [LC]', '卵白 [LC]', '卵白 [UA/mL]', 'ソバ [LC]', 
        'ピーナッツ [LC]', 'ラテックス [LC]',
        '抗体濃度 [U/mL]',
        'IgE（非特異的IgE） [IU/mL]'],

    'kidney': [
        'アルブミン／Cre補正 [mg/g.cre]',
        'アルブミン定量値 [μg/mL]',
        'クレアチニン（尿中） [g/L]',
        'Na（ナトリウム）（尿中） [g/L]',
        'Cl（クロール）（尿中） [g/L]',
        'K（カリウム）（尿中） [g/L]',
        'シスタチンC［金コロイド凝集法］ [mg/L]',
        'GFR区分（シスタチンC）[ml/min/1.73㎡]',],
    
    'administrative': [
        '受診日',
        'イベントコード',
        'コースコード','検査会社']}

# 3. 特徴量別人数把握

In [ ]:

tempDF = testDF.copy()


for category, features in test_dict.items():

# laboDFに実際に存在する特徴量だけ使用
    features = [col for col in features if col in tempDF.columns]

    # 特徴量ごとの欠損のない人数
    resultDF = pd.DataFrame({'Feature': features,
                                'N': len(tempDF),
                                 'Non-missing N': [tempDF[col].notna()
                                     .sum() for col in features],
                                'missing N': [tempDF[col].isna().sum() for col in features]})
    resultDF=resultDF.sort_values('Non-missing N',ascending=False)

    print(f'===== {category} =====')
    display(resultDF)

# 4. regression

## -1 prepare DF

In [ ]:
modelD = {
    'MetAge': metDF,
    'AntAge': antDF,
    'HemAge': hemDF,
    'ChemAge': chemDF,
    'LabAge': labDF}

modelDs = {
    'MetAge': metDF,
    'AntAge': antDF,
    'HemAge': hemDF,
    'ChemAge': chemDF,
    'LabAge': labDF,
    'AlgAge':algDF,
    'KidAge':kidDF}

for modelDF in modelDs.values():
    for cohort in cohortD:
        modelDF[cohort]['Sex'] = demoDF.loc[modelDF[cohort].index, 'Sex']
        modelDF[cohort]['BMI'] = demoDF.loc[modelDF[cohort].index, 'BMI']

display(metDF['SexStratified'])

## -2 fit regression

In [ ]:
# age, sex, BMI
# Regression and FDR correction by panel

resultD = {}

for model, modelDF in modelDs.items():
    cohort = 'SexStratified'

    tempDF1 = testDF
    baseDF = modelDF[cohort].copy()

    results = []

        # ==========================================
        # Regression: panelごとに実施
        # ==========================================
    for panel, features in test_dict.items():

        panel_results = []

        for col in features:

            if col not in tempDF1.columns:
                continue

                # データ結合
            tempDF = baseDF.join(
                    tempDF1[[col]],
                    how='inner')

            temp = tempDF[[col, 'AgeGap', 'Age', 'Sex', 'BMI']].dropna().copy()

                # 検査項目が一定値しかない場合はスキップ
            if temp[col].nunique() <= 1:
                 continue

            # Z-score
            sd = temp[col].std()
            if pd.isna(sd) or sd == 0:
                continue
            temp[col] = (temp[col] - temp[col].mean()) / sd

                # Regression
            formula = (f'Q("{col}") ~ AgeGap + Age + BMI + C(Sex)')

            fit = smf.ols(
                    formula,
                    data=temp).fit()

                # 95% CI
            ci = fit.conf_int().loc['AgeGap']

            panel_results.append({
                    'Panel': panel,
                    'Variable': col,
                    'beta': fit.params['AgeGap'],
                    'CI_low': ci[0],
                    'CI_high': ci[1],
                    'p': fit.pvalues['AgeGap'],
                    'n': len(temp) })

            # ==========================================
            # FDR: panel内のphenotypeのみで補正
            # ==========================================
        panelDF = pd.DataFrame(panel_results)

        if not panelDF.empty:

            panelDF['p_FDR'] = multi.multipletests(
                    panelDF['p'],
                    method='fdr_bh')[1]

            panelDF['significant'] = (
                    panelDF['p_FDR'] < 0.05)

            results.append(panelDF)

        # ==========================================
        # 全panelの結果を結合して保存
        # ==========================================
    if results:
        resultDF = pd.concat(
                results,
                ignore_index=True)
    else:
        resultDF = pd.DataFrame(
                columns=[
                    'Panel', 'Variable', 'beta',
                    'CI_low', 'CI_high', 'p',
                    'n', 'p_FDR', 'significant'])

    print(f'{model} - {cohort}')
    display(resultDF)

    resultD[(model, cohort)] = resultDF.copy()

        # ==========================================
        # Panel別の描画
        # ==========================================
    for panel in resultDF['Panel'].unique():

        plotDF = (
                resultDF[resultDF['Panel'] == panel]
                .sort_values('beta')
                .reset_index(drop=True))

        if plotDF.empty:
            continue

        plt.figure(figsize=(10, max(4, len(plotDF) * 0.35)))

        for i, row in plotDF.iterrows():
             if row['significant']:
                color = 'tab:red'
                alpha = 1.0
             else:
                color = 'gray'
                alpha = 0.3

             plt.errorbar(
                    row['beta'],
                    i,
                    xerr=[[
                        row['beta'] - row['CI_low']
                    ], [
                        row['CI_high'] - row['beta']
                    ]],
                    color=color,
                    capsize=4,
                    elinewidth=2,
                    alpha=alpha)

        plt.axvline(
                0,
                linestyle='--',
                linewidth=1)

        plt.yticks(
                range(len(plotDF)),
                plotDF['Variable'])

        plt.xlabel('Beta (95% CI)')
        plt.title(f'{model} - {cohort} - {panel}')

        plt.tight_layout()
        plt.show()

In [ ]:
model_colors = {
    'MetAge': 'tab:blue',
    'AntAge': 'tab:orange',
    'HemAge': 'tab:green',
    'ChemAge': 'tab:red',
    'LabAge': 'tab:purple',
    'AlgAge': 'tab:brown',
    'KidAge': 'tab:pink'}

In [ ]:
model_order = list(modelDs.keys())

cohort = 'SexStratified'

# 循環が起きるモデルとpanelの対応
excludeD = {
    'allergy': 'AlgAge',
    'kidney': 'KidAge'}

# panelごとに描画
for panel, features in test_dict.items():

    # ==========================================
    # 各モデルの結果を結合
    # ==========================================
    plotD = []

    for model in model_order:

        temp = resultD[(model, cohort)].copy()

        # 対象panelのみ
        temp = temp[temp['Panel'] == panel].copy()

        # 循環が起きるモデルを除外
        if excludeD.get(panel) == model:
            continue

        temp['Model'] = model
        plotD.append(temp)

    if not plotD:
        continue

    plotDF = pd.concat(
        plotD,
        ignore_index=True)

    if plotDF.empty:
        continue

    # ==========================================
    # phenotypeの並び順
    # panel内で平均betaの小さい順
    # ==========================================
    disease_order = (
        plotDF.groupby('Variable')['beta']
        .mean()
        .sort_values()
        .index.tolist())

    if len(disease_order) == 0:
        continue

    # ==========================================
    # Plot
    # ==========================================
    disease_spacing = 2.0

    y_positions = {
        disease: i * disease_spacing
        for i, disease in enumerate(disease_order)}

    # モデルの位置を固定
    offsets = np.linspace(
        -0.6,
        0.6,
        len(model_order))

    fig, ax = plt.subplots(
        figsize=(20, max(7, len(disease_order) * 0.7)))

    for offset, model in zip(offsets, model_order):

        # 循環が起きるモデルは除外
        if excludeD.get(panel) == model:
            continue

        temp = plotDF[plotDF['Model'] == model].copy()

        # phenotypeの順番を揃える
        temp = (
            temp.set_index('Variable')
            .reindex(disease_order)
            .reset_index())

        for _, row in temp.iterrows():

            if pd.isna(row['beta']):
                continue

            # 有意なら濃く、非有意なら薄く
            alpha = 1.0 if row['significant'] else 0.3

            ax.errorbar(
                row['beta'],
                y_positions[row['Variable']] + offset,
                xerr=[[
                    row['beta'] - row['CI_low']
                ], [
                    row['CI_high'] - row['beta']
                ]],
                fmt='o',
                color=model_colors[model],
                ecolor=model_colors[model],
                capsize=3,
                elinewidth=1.5,
                alpha=alpha)

    # ==========================================
    # 軸・凡例
    # ==========================================
    ax.axvline(
        0,
        linestyle='--',
        linewidth=1,
        color='gray')

    ax.set_yticks([y_positions[d] for d in disease_order])
    ax.set_yticklabels(disease_order, fontsize=28)
    ax.set_xlabel('Beta (95% CI)',fontsize=28)
    ax.set_ylabel('Phenotype',fontsize=28)
    ax.tick_params(axis='both', labelsize=24)
    ax.text(0.5, -0.35, f'{cohort}: {panel} - Model comparison',
           transform=ax.transAxes,
           ha='center',
           va='top',
           fontsize=30)

    # 実際に描画対象となるモデルだけ凡例に表示
    legend_models = [
        model for model in model_order
        if excludeD.get(panel) != model]

    legend_handles = [
        Line2D(
            [0], [0],
            color=model_colors[model],
            marker='o',
            linestyle='-',
            label=model)
        for model in legend_models]

    ax.legend(
        handles=legend_handles,
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        title='Model',
        fontsize=24)

    plt.tight_layout()

    #plt.savefig( f'{cohort}_{panel}_Model_comparison.png',bbox_inches='tight',dpi=300)

    plt.show()

サイズ調整

In [ ]:
model_order = list(modelDs.keys())

cohort = 'SexStratified'

# ==========================================
# Settings：ここを変更して調整
# ==========================================

# 図のサイズ（inch）
fig_width = 20
fig_height_per_phenotype = 0.9
fig_height_min = 8

# phenotype 間の縦方向の間隔
disease_spacing = 2.8

# モデル間の縦方向のオフセット
model_offset_range = 0.95

# CI とプロット
ci_linewidth = 2.8
ci_capsize = 5
ci_capthick = 2.5
marker_size = 9

# 有意・非有意の透明度
alpha_sig = 1.0
alpha_nonsig = 0.4

# フォントサイズ（pt）
phenotype_fontsize = 24
tick_fontsize = 22
axis_label_fontsize = 26
title_fontsize = 28
legend_fontsize = 20
legend_title_fontsize = 22

# 0ライン
zero_linewidth = 2.0

# 保存解像度
save_dpi = 300


# 循環が起きるモデルとpanelの対応
excludeD = {
    'allergy': 'AlgAge',
    'kidney': 'KidAge'
}


# ==========================================
# Panelごとに描画
# ==========================================

for panel, features in test_dict.items():

    # ------------------------------------------
    # 各モデルの結果を結合
    # ------------------------------------------
    plotD = []

    for model in model_order:

        temp = resultD[(model, cohort)].copy()
        temp = temp[temp['Panel'] == panel].copy()

        # 循環が起きるモデルを除外
        if excludeD.get(panel) == model:
            continue

        temp['Model'] = model
        plotD.append(temp)

    if not plotD:
        continue

    plotDF = pd.concat(plotD, ignore_index=True)

    if plotDF.empty:
        continue

    # ------------------------------------------
    # phenotypeの並び順
    # panel内で平均betaの小さい順
    # ------------------------------------------
    disease_order = (
        plotDF.groupby('Variable')['beta']
        .mean()
        .sort_values()
        .index.tolist()
    )

    if len(disease_order) == 0:
        continue

    # ------------------------------------------
    # 描画対象モデル
    # ------------------------------------------
    legend_models = [
        model for model in model_order
        if excludeD.get(panel) != model
    ]

    n_models = len(legend_models)
    n_diseases = len(disease_order)

    # ------------------------------------------
    # Y位置
    # ------------------------------------------
    y_positions = {
        disease: i * disease_spacing
        for i, disease in enumerate(disease_order)
    }

    # 実際に描画するモデル数に応じてオフセットを設定
    offsets = np.linspace(
        -model_offset_range,
        model_offset_range,
        n_models
    ) if n_models > 1 else np.array([0.0])

    offsetD = dict(zip(legend_models, offsets))

    # ------------------------------------------
    # 図のサイズを自動調整
    # ------------------------------------------
    fig_height = max(
        fig_height_min,
        n_diseases * fig_height_per_phenotype
    )

    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height)
    )

    # ------------------------------------------
    # 各モデルの描画
    # ------------------------------------------
    for model in legend_models:

        offset = offsetD[model]
        temp = plotDF[plotDF['Model'] == model].copy()

        temp = (
            temp.set_index('Variable')
            .reindex(disease_order)
            .reset_index()
        )

        for _, row in temp.iterrows():

            if pd.isna(row['beta']):
                continue

            # CIの値がそろっている行だけ描画
            if pd.isna(row['CI_low']) or pd.isna(row['CI_high']):
                continue

            # 有意なら濃く、非有意なら薄く
            alpha = (
                alpha_sig
                if row['significant']
                else alpha_nonsig
            )

            ax.errorbar(
                x=row['beta'],
                y=y_positions[row['Variable']] + offset,
                xerr=[[
                    max(0, row['beta'] - row['CI_low'])
                ], [
                    max(0, row['CI_high'] - row['beta'])
                ]],
                fmt='o',
                color=model_colors[model],
                ecolor=model_colors[model],
                markersize=marker_size,
                markeredgewidth=1.5,
                capsize=ci_capsize,
                capthick=ci_capthick,
                elinewidth=ci_linewidth,
                linewidth=ci_linewidth,
                alpha=alpha,
                zorder=3
            )

    # ------------------------------------------
    # 軸設定
    # ------------------------------------------
    ax.axvline(
        0,
        linestyle='--',
        linewidth=zero_linewidth,
        color='gray',
        alpha=0.9,
        zorder=1
    )

    ax.set_yticks([
        y_positions[d] for d in disease_order
    ])

    ax.set_yticklabels(
        disease_order,
        fontsize=phenotype_fontsize
    )

    ax.set_xlabel(
        'Beta (95% CI)',
        fontsize=axis_label_fontsize,
        labelpad=12
    )

    ax.set_ylabel(
        'Phenotype',
        fontsize=axis_label_fontsize,
        labelpad=12
    )

    # 目盛りラベル・目盛り線
    ax.tick_params(
        axis='both',
        labelsize=tick_fontsize,
        width=1.8,
        length=7
    )

    # phenotypeごとのモデル間隔を維持
    ax.set_ylim(
        -disease_spacing * 0.8,
        (n_diseases - 1) * disease_spacing
        + disease_spacing * 0.8
    )

    # ------------------------------------------
    # 下部タイトル
    # ------------------------------------------
    ax.text(
        0.5,
        -0.22,
        f'{cohort}: {panel} - Model comparison',
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=title_fontsize
    )

    # ------------------------------------------
    # 凡例
    # ------------------------------------------
    legend_handles = [
        Line2D(
            [0], [0],
            color=model_colors[model],
            marker='o',
            markersize=marker_size,
            linewidth=ci_linewidth,
            label=model
        )
        for model in legend_models
    ]

    ax.legend(
        handles=legend_handles,
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        title='Model',
        fontsize=legend_fontsize,
        title_fontsize=legend_title_fontsize,
        frameon=False,
        handlelength=2.0,
        labelspacing=0.8,
        borderaxespad=0
    )

    # ------------------------------------------
    # レイアウト
    # ------------------------------------------
    fig.subplots_adjust(
        left=0.30,
        right=0.78,
        top=0.96,
        bottom=0.16
    )


    plt.show()
    plt.close(fig)

## -2 age stratification

In [ ]:
age_bins = [20, 30, 40, 50, 60, 70, 80, 90]
age_labels = ['20-29', '30-39', '40-49', '50-59','60-69', '70-79', '80-89']

tempD = {}
cohort = 'SexStratified'
for model,modelDF in modelDs.items():
    tempDF = modelDF[cohort].copy()

    tempDF['AgeGroup'] = pd.cut(
    tempDF['Age'],
    bins=age_bins,
    labels=age_labels,
    right=False)

    print(model)
    display(tempDF)
    tempD[model] = tempDF

models = tempD

In [ ]:
# age, sex, BMI
age_groups = age_labels
reference_group = '40-49'
resultD = {} 
# 循環が起きるモデルとpanelの対応
excludeD = {
    'allergy': 'AlgAge',
    'kidney': 'KidAge'}
for model, modelDF in models.items():
    tempDF1 = testDF
    baseDF = modelDF.copy()
    results = []
    for panel, features in test_dict.items():
        if model in excludeD.get(panel, []):
            continue
        panels = []

        for col in features:
            if col not in tempDF1.columns:
                continue

            tempDF = baseDF.join(tempDF1[[col]],how='inner')
            temp = tempDF[[col, 'AgeGap', 'AgeGroup', 'Sex','BMI']].dropna()


                # 検査項目が一定値しかない場合
            if temp[col].nunique() <= 1:
                continue
            sd = temp[col].std()
            if pd.isna(sd) or sd == 0:
                continue
                
                #Z-score
            temp[col] = (temp[col] - temp[col].mean()) / temp[col].std()
        
            formula = f'Q("{col}") ~ AgeGap * C(AgeGroup, Treatment(reference="40-49")) + BMI + C(Sex)'

            try:
                fit = smf.ols(formula,data=temp).fit()

            except Exception as e:
                print(f'Skip:{model}/{panel}/{col}/{e}')
                continue

            params = fit.params
            cov = fit.cov_params()
            param_names = list(params.index)

            main_term = 'AgeGap'

            if main_term not in param_names:
                continue

            # 各年齢層の傾きを線形コントラストで計算
            for age_group in age_groups:
                if age_group not in temp['AgeGroup'].astype(str).values:
                    continue

                # β = 基準のβ + interaction係数
                contrast = np.zeros(len(param_names))
                contrast[param_names.index(main_term)] = 1.0
                if age_group != reference_group:

                    interaction_term = next(
                        (name for name in param_names 
                         if name.startswith('AgeGap:') 
                         and name.endswith(f'[T.{age_group}]')),None)

                    if interaction_term is None:
                        print(f'Interaction not found: '
                            f'{model} / {col} / {age_group}')
                        continue

                    contrast[param_names.index(interaction_term)] = 1.0

                # 線形コントラスト：推定値、SE、95% CI、p値
                test_result = fit.t_test(contrast)
                beta = float(np.asarray(test_result.effect).ravel()[0])
                ci = np.asarray(test_result.conf_int()).reshape(-1, 2)[0]
                p = float(np.asarray(test_result.pvalue).ravel()[0])
            
                panels.append({
                    'Panel': panel,
                    'Variable': col,
                    'AgeGroup': age_group,
                    'beta': beta,
                    'CI_low': ci[0],
                    'CI_high': ci[1],
                    'p': p,
                    'n': len(temp)})

        panelDF = pd.DataFrame(panels)
        if not panelDF.empty:
        # FDR
            panelDF['p_FDR'] = multi.multipletests(panelDF['p'], method='fdr_bh')[1]
            panelDF['significant'] = panelDF['p_FDR'] < 0.05
            results.append(panelDF)

    if results:
        resultDF = pd.concat(results, ignore_index=True)
    else:
        resultDF = pd.DataFrame()
    print(f'{model}')
    display(resultDF)
    resultD[model] = resultDF.copy()

In [ ]:
# 年齢層ごとの色
agegroup_colorD = {
    '20-29': 'tab:purple',
    '30-39': 'tab:blue',
    '40-49': 'tab:green',
    '50-59': 'tab:orange',
    '60-69': 'tab:red',
    '70-79': 'tab:brown',
    '80-89': 'tab:gray'}

for model in models.keys():

    resultDF = resultD.get(model, pd.DataFrame())

    if resultDF.empty:
        print(f'Skip: {model} (no results)')
        continue

    for panel in resultDF['Panel'].unique():

        plotDF = resultDF[resultDF['Panel'] == panel].copy()

        if plotDF.empty:
            continue

        # 検査項目の表示順
        variables = plotDF['Variable'].drop_duplicates().tolist()
        y_base = {var: i for i, var in enumerate(variables)}

        # 年齢層ごとの縦位置のずらし幅
        offsets = np.linspace(-0.3, 0.3, len(age_groups))

        fig, ax = plt.subplots(
            figsize=(8, max(3, len(variables) * 0.45))
        )

        for i, age_group in enumerate(age_groups):

            tempDF = plotDF[plotDF['AgeGroup'] == age_group].copy()

            if tempDF.empty:
                continue

            if age_group not in agegroup_colorD:
                continue
            color = agegroup_colorD[age_group]

            y = np.array([
                y_base[var] + offsets[i]
                for var in tempDF['Variable']])

            ax.errorbar(
                tempDF['beta'],
                y,
                xerr=[
                    tempDF['beta'] - tempDF['CI_low'],
                    tempDF['CI_high'] - tempDF['beta']
                ],
                fmt='none',
                ecolor=color,
                elinewidth=1,
                capsize=2,
                zorder=1)

            # FDR補正後の有意性で分けて描画
            sig = tempDF['significant'].fillna(False).astype(bool)

    # 非有意：年齢層の色を薄く表示
            ax.scatter(
        tempDF.loc[~sig, 'beta'],
        y[~sig.to_numpy()],
        color=color,
        alpha=0.25,
        s=25,
        zorder=2)

    # 有意：年齢層の色を濃く表示
            ax.scatter(
        tempDF.loc[sig, 'beta'],
        y[sig.to_numpy()],
        color=color,
        alpha=1.0,
        s=35,
        label=f'{age_group} (FDR < 0.05)',
        zorder=3)

        # β = 0 の基準線
        ax.axvline(0, color='black', linestyle='--', linewidth=1)

        ax.set_yticks(list(y_base.values()))
        ax.set_yticklabels(list(y_base.keys()))

        ax.set_xlabel('Regression coefficient (β)')
        ax.set_ylabel('Phenotype')
        ax.set_title(f'{model} - {panel}: Age-specific slopes')

        ax.legend(
            title='Age group',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            borderaxespad=0
        )

        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()

In [ ]:
# ==========================================
# Settings
# ==========================================
cohort = 'SexStratified'

agegroup_order = [
    '20-29', '30-39', '40-49',
    '50-59', '60-69', '70-79', '80-89'
]

model_order = list(models.keys())

# ==========================================
# ModelごとにHeatmapを作成
# ==========================================
for model in model_order:

    key = model

    if key not in resultD or resultD[key].empty:
        print(f'Skip {model}: no results')
        continue

    plotDF = resultD[key].copy()

    # --------------------------------------
    # Panelごとに処理
    # --------------------------------------
    for panel in plotDF['Panel'].dropna().unique():

        panelDF = plotDF[plotDF['Panel'] == panel].copy()

        if panelDF.empty:
            continue

        # --------------------------------------
        # beta行列：行=Phenotype、列=AgeGroup
        # --------------------------------------
        beta_mat = panelDF.pivot_table(
            index='Variable',
            columns='AgeGroup',
            values='beta',
            aggfunc='first'
        )

        # 年齢層の順番を統一
        beta_mat.columns = beta_mat.columns.astype(str)
        beta_mat = beta_mat.reindex(
            columns=[
                ag for ag in agegroup_order
                if ag in beta_mat.columns
            ]
        )

        if beta_mat.empty or beta_mat.shape[1] == 0:
            continue

        # --------------------------------------
        # FDR有意性の行列
        # --------------------------------------
        sig_mat = panelDF.pivot_table(
            index='Variable',
            columns='AgeGroup',
            values='significant',
            aggfunc='first'
        )

        sig_mat.columns = sig_mat.columns.astype(str)
        sig_mat = sig_mat.reindex(
            index=beta_mat.index,
            columns=beta_mat.columns
        )

        # --------------------------------------
        # 共通の色スケール
        # --------------------------------------
        all_beta = beta_mat.to_numpy(dtype=float)
        all_beta = all_beta[np.isfinite(all_beta)]

        vmax = np.max(np.abs(all_beta)) if len(all_beta) else 1
        if vmax == 0:
            vmax = 1

        # --------------------------------------
        # Heatmap
        # --------------------------------------
        fig_width = max(6, 1.0 * beta_mat.shape[1])
        fig_height = max(4, 0.35 * len(beta_mat.index))

        fig, ax = plt.subplots(
            figsize=(fig_width, fig_height)
        )

        sns.heatmap(
            beta_mat,
            ax=ax,
            cmap='RdBu_r',
            center=0,
            vmin=-vmax,
            vmax=vmax,
            linewidths=0.3,
            linecolor='lightgray',
            cbar_kws={
                'label': 'Beta (AgeGap_reg)'
            }
        )

        # FDR有意なセルに黒枠を付ける
        for i, variable in enumerate(beta_mat.index):
            for j, agegroup in enumerate(beta_mat.columns):

                if (
                    variable in sig_mat.index
                    and agegroup in sig_mat.columns
                    and pd.notna(sig_mat.loc[variable, agegroup])
                    and bool(sig_mat.loc[variable, agegroup])
                ):
                    ax.add_patch(
                        plt.Rectangle(
                            (j, i), 1, 1,
                            fill=False,
                            edgecolor='black',
                            linewidth=1.5
                        )
                    )

        ax.set_title(f'{model} | {panel} | {cohort}')
        ax.set_xlabel('Age group')
        ax.set_ylabel('Phenotype')
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)

        plt.tight_layout()
        plt.show()

In [ ]:
# ==========================================
# Settings
# ==========================================
cohort = 'SexStratified'

agegroup_order = [
    '20-29', '30-39', '40-49',
    '50-59', '60-69', '70-79', '80-89']

model_order = list(models.keys())
panel_order = list(test_dict.keys())

# ==========================================
# Panelごとに、横7列のHeatmapを作成
# ==========================================
for panel in panel_order:

    # --------------------------------------
    # 各モデルのpanel結果を取得
    # --------------------------------------
    betaD = {}
    sigD = {}

    for model in model_order:

        key = model

        if key not in resultD or resultD[key].empty:
            continue

        tempDF = resultD[key].copy()
        tempDF = tempDF[tempDF['Panel'] == panel].copy()

        if tempDF.empty:
            continue

        # beta matrix: rows=Phenotype, columns=AgeGroup
        beta_mat = tempDF.pivot_table(
            index='Variable',
            columns='AgeGroup',
            values='beta',
            aggfunc='first')

        beta_mat.columns = beta_mat.columns.astype(str)
        beta_mat = beta_mat.reindex(
            columns=[
                ag for ag in agegroup_order
                if ag in beta_mat.columns])

        if beta_mat.empty:
            continue

        # FDR significance matrix
        sig_mat = tempDF.pivot_table(
            index='Variable',
            columns='AgeGroup',
            values='significant',
            aggfunc='first')

        sig_mat.columns = sig_mat.columns.astype(str)
        sig_mat = sig_mat.reindex(
            index=beta_mat.index,
            columns=beta_mat.columns)

        betaD[model] = beta_mat
        sigD[model] = sig_mat

    if not betaD:
        print(f'Skip {panel}: no results')
        continue

    # --------------------------------------
    # 全モデルでPhenotypeの順番を統一
    # --------------------------------------
    variable_order = list(dict.fromkeys(
        variable
        for model in model_order
        if model in betaD
        for variable in betaD[model].index))

    for model in betaD:

        betaD[model] = betaD[model].reindex(
            index=variable_order,
            columns=[
                ag for ag in agegroup_order
                if ag in betaD[model].columns])

        sigD[model] = sigD[model].reindex(
            index=variable_order,
            columns=betaD[model].columns)

    # --------------------------------------
    # 全7モデル共通の色スケール
    # --------------------------------------
    all_beta = np.concatenate([
        betaD[model].to_numpy(dtype=float).ravel()
        for model in betaD])

    all_beta = all_beta[np.isfinite(all_beta)]

    vmax = np.max(np.abs(all_beta)) if len(all_beta) else 1

    if vmax == 0:
        vmax = 1

    # --------------------------------------
    # 1 row × 7 columns
    # --------------------------------------
    ncols = len(model_order)
    n_variables = len(variable_order)

    # 表現型の数に応じて図の高さを変更
    fig_height = max(2.5, 0.42 * n_variables)

    fig, axes = plt.subplots(
        nrows=1,
        ncols=ncols,
        figsize=(3.0 * ncols, fig_height),
        squeeze=False)

    # --------------------------------------
    # 各モデルのHeatmap
    # --------------------------------------
    for c, model in enumerate(model_order):

        ax = axes[0, c]

        if model not in betaD:
            ax.axis('off')
            continue

        mat = betaD[model]
        sig_mat = sigD[model]

        sns.heatmap(
            mat,
            ax=ax,
            cmap='RdBu_r',
            center=0,
            vmin=-vmax,
            vmax=vmax,
            linewidths=0.2,
            linecolor='lightgray',
            cbar=False)

        # ----------------------------------
        # FDR有意なセルを黒枠で表示
        # ----------------------------------
        for i, variable in enumerate(mat.index):
            for j, agegroup in enumerate(mat.columns):

                value = sig_mat.loc[variable, agegroup]

                if pd.notna(value) and bool(value):
                    ax.add_patch(
                        plt.Rectangle(
                            (j, i),
                            1,
                            1,
                            fill=False,
                            edgecolor='black',
                            linewidth=1))

        # ----------------------------------
        # 軸ラベル・タイトル
        # ----------------------------------
        ax.set_title(model, fontsize=12, pad=10)
        ax.set_xlabel('Age group', fontsize=9)

        ax.tick_params(
            axis='x',
            labelrotation=90,
            labelsize=7)

        ax.tick_params(
            axis='y',
            labelsize=7)

        if c == 0:
            ax.set_ylabel('Phenotype', fontsize=10)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)

    # --------------------------------------
    # 共通カラーバー
    # --------------------------------------
    fig.subplots_adjust(
        left=0.08,
        right=0.92,
        bottom=0.18,
        top=0.80,
        wspace=0.12)

    # 最後のヒートマップと同じ高さに配置
    pos = axes[0, -1].get_position()

    cbar_ax = fig.add_axes([
        0.94,
        pos.y0,
        0.012,
        pos.height])

    norm = plt.Normalize(vmin=-vmax, vmax=vmax)
    sm = plt.cm.ScalarMappable(
        cmap='RdBu_r',
        norm=norm)
    sm.set_array([])

    fig.colorbar(
        sm,
        cax=cbar_ax,
        label='Beta (AgeGap_reg)')

    fig.suptitle(
        f'{cohort} | {panel}: Age-specific beta by model',
        fontsize=15,
        y=0.99)

    plt.show()

# 6. disease regression

## -1 import and prepare

In [ ]:
fileDir = './ExportData/Preprocess/ffq/'
ipynbName = '261008_TMM-BAge_CommCohort67K-ver1_'
fileName = 'life-disease.tsv'
tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))
diDF=tempDF

In [ ]:
#paperの16疾患
#モデル関連しそうな疾患
#代謝に関連する疾患に絞る

# 本人の既往歴項目：パネル別
history_dict = {
    'tier1':{
        # previous paper 16 chronical disease
        'Cancer': [
            '胃がん',
            '大腸がん',
            '肺がん',
            '肝がん',
            '腎臓がん',
            '膵臓がん',
            '皮膚がん',
            '乳がん',
            '精巣がん',
            '前立腺がん',
            '脳腫瘍',
            '白血病',
            '悪性リンパ腫',
            '多発性骨髄腫',
            '卵巣がん',
            '子宮頚がん',
            '子宮体がん'],

        'Diabetes':[
            '１型糖尿病',
            '２型糖尿病'],

        'Hypertension':['高血圧'],

        'IHD':['心筋梗塞・狭心症'],

        'Stroke':[
            '脳出血',
            '脳梗塞',
            'くも膜下出血'],

        'CKD_relate':[
            '慢性腎炎（IgA腎症、糸球体腎炎）',
            '腎透析',
            'ネフローゼ症候群'],

        'COPD':['慢性閉塞性肺疾患（COPD）'],

        'Depression':[
            'うつ病 震災前',
            'うつ病 震災後'],

        'Bipolar':['双極性障害（躁うつ病）'],

        'Schizophrenia':['統合失調症'],

        'Demantia':['認知症'],

        'Parkinsonism':['パーキンソン病'],

        'Osteoarthritis':['変形性膝関節症',],

        #'Osteoporosis':[],　疾患項目なし
        #'Multiple sclerosis':[],　疾患項目なし
        #'Cirrhosis':[],

    #maybe metage and speage related
        'anemia':['鉄欠乏性貧血'],
        'dyslipidemia':['高脂血症'],
        'fatty_liver':['脂肪肝'],
        'hepatitis':['B型肝炎',
                     'C型肝炎'],
        'urinary_calculus':['尿管結石、腎結石'],
        'allergic_disease':['食物アレルギー',
                           'アレルギー性鼻炎',
                           'アレルギー性結膜炎',
                           'アトピー性皮膚炎',
                           'じんましん',
                           '気管支ぜん息'],
        'autoimmune':['膠(こう)原(げん)病',
                      '自己免疫性疾患',
                      '全身性エリテマトーデス',
                      'リウマチ']},

    'tier3':{
        'infectious_disease': ['結核'],
        'endocrine_metabolic': [
            '甲状腺機能亢進症・バセドウ氏病',
            '甲状腺機能低下症・橋本病',
            '高尿酸血症（痛風）',],

        'mental_behavioral': [
            '不安障害',
            'PTSD（心的外傷後ストレス障害） 震災前',
            'PTSD（心的外傷後ストレス障害） 震災後',
            '注意欠陥・多動性障害（ADHD）',
            '学習障害（LD）',
            '広汎性発達障害（自閉症、 アスペルガー症候群含む）',
            '神経性無食欲症（拒食症）',
            '神経性過食（大食）症'],

        'neurological': [
            'てんかん',
            '偏頭痛',
            '髄膜炎・脳炎',
            '水頭症'],

        'eye_ear': [
            '近視',
            '遠視',
            '乱視',
            '緑内障',
            '黄斑変性',
            '角膜疾患',
            '難聴',
            '慢性中耳炎',],

        'circulatory': [
            '動脈瘤・大動脈解離',
            '心不全',
            'マルファン症候群',
            '心房細動',
            'ペースメーカー',
            '心室細動（除細動器）'],

        'respiratory': [
            '慢性気管支炎',
            '慢性副鼻腔炎'],

        'digestive': [
            '逆流性食道炎',
            '胃炎',
            '胃潰瘍',
            '十二指腸潰瘍',
            '過敏性腸症候群',
            'クローン病',
            '潰瘍性大腸炎',
            '胆石',
            '膵炎'],

        'skin': [
            '薬疹・薬剤アレルギー',
            '接触性皮膚炎',
            '乾癬'],

        'musculoskeletal': [
            '脊椎側弯(そくわん)症',
            '川崎病',],

        'female_disease': [
            '妊娠高血圧症候群（妊娠中毒症）',
            '妊娠糖尿病',
            '子宮内膜症',
            '子宮筋腫'],

        'congenital_chromosomal': [
            '先天性心疾患',
            '先天性股関節脱臼'],

        'injury_external_cause': [
            '腰の骨折',
            '腕か手首の骨折',
            '大腿骨（太ももの骨）の付け根の骨折',
            '意識喪失を伴う頭部外傷'],

        'other_disease': [
            'シックハウス症候群',
            '化学物質過敏症' ],

        'dental_maxillofacial': [
            'う歯（むし歯）',
            '歯周病',
            '上顎前突',
            '下顎前突',
            '唇・顎・口蓋裂']}}

## -3 regression

### logistic

In [ ]:
#症例数が少なすぎる場合はスキップ

# age, sex, BMI
resultD = {}
skipL =[]

cohort = 'SexStratified'

for model, modelDF in modelDs.items():
    baseDF = modelDF[cohort].copy()
    results = []

    for tier in ['tier1']:
        for disease, keywords in history_dict[tier].items():
                # history_dictの項目名を含むdiDFのcolumnを検索
            matched_cols = []
            for keyword in keywords:
                matched_cols.extend([
                        col for col in diDF.columns
                        if keyword in col])

                # 重複削除
            matched_cols = list(dict.fromkeys(matched_cols))

            if len(matched_cols) == 0:
                continue

                # 疾患群を0/1にまとめる
            diseaseDF = pd.DataFrame(index=diDF.index)

            diseaseDF[disease] = (
                    diDF[matched_cols]
                    .eq(1)
                    .any(axis=1)
                    .astype(float))

                # 全項目がNaNの場合はNaN
            all_nan = diDF[matched_cols].isna().all(axis=1)
            diseaseDF.loc[all_nan, disease] = np.nan

                # modelDFと結合
            tempDF = baseDF.join(
                    diseaseDF[[disease]],
                    how='inner')

            temp = tempDF[[disease, 'AgeGap', 'Age', 'Sex', 'BMI']].dropna()

                # 発症歴が一定値しかない場合
            if temp[disease].nunique() <= 1:
                continue

            cases = (temp[disease] == 1).sum()
            controls = (temp[disease] == 0).sum()

            if cases < 10 or controls < 10:
                skipL.append({
                        'Disease': disease,
                        'Model': model,
                        'Cohort': cohort,
                        'Cases': int(cases),
                        'Controls': int(controls),
                        'n': len(temp)})
                print(f'Skip: {model} / {cohort} / {disease} '
                        f'(Cases={cases}, Controls={controls})')
                continue

            formula = (f'Q("{disease}") ~ AgeGap + Age + BMI + C(Sex)')

            fit = smf.logit(
                    formula,
                    data=temp).fit(disp=False)

                # 95% CI
            coef = fit.params['AgeGap']
            ci = fit.conf_int().loc['AgeGap']

            results.append({
                    'Tier': tier,
                    'Variable': disease,
                    'Cases': int((temp[disease] == 1).sum()),
                    'Controls': int((temp[disease] == 0).sum()),
                    'OR': np.exp(coef),
                    'CI_low': np.exp(ci[0]),
                    'CI_high': np.exp(ci[1]),
                    'p': fit.pvalues['AgeGap'],
                    'n': len(temp)})

    resultDF = pd.DataFrame(results)

        # FDR
    if len(resultDF) > 0:
        resultDF['p_FDR'] = multi.multipletests(
                resultDF['p'],
                method='fdr_bh')[1]

        resultDF['significant'] = resultDF['p_FDR'] < 0.05

    print(cohort)
    display(resultDF)
    resultD[(model, cohort)] = resultDF.copy()


        # plot
    plotDF = (resultDF[resultDF['Tier'] == tier]
                .sort_values('OR')
                .reset_index(drop=True))

    if len(plotDF) == 0:
        continue

    plt.figure(figsize=(10, max(4, len(plotDF) * 0.35)))

    for i, row in plotDF.iterrows():
        if row['significant']:
            color = 'tab:red'
            alpha = 1.0
        else:
            color = 'gray'
            alpha = 0.3

        plt.errorbar(
                    row['OR'],
                    i,
                    xerr=[[
                        row['OR'] - row['CI_low']], [
                        row['CI_high'] - row['OR']]],
                    color=color,
                    capsize=4,
                    elinewidth=2,
                    alpha=alpha)

    plt.axvline(
                1,
                linestyle='--',
                linewidth=1)

    plt.yticks(
                range(len(plotDF)),
                plotDF['Variable'])

    plt.xlabel('ODDs Ratio (95% CI)')
    plt.title(f'{model} - {cohort} - disease')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# Skipされた疾患を疾患ごとに集計
# ==========================================

skipDF = pd.DataFrame(skipL)

if len(skipDF) == 0:
    print('All disease ditected.')

else:
    for disease, tempDF in skipDF.groupby('Disease', sort=True):

        print(f'\n{disease}')

        for _, row in tempDF.iterrows():

            print(
                f"  {row['Model']} / {row['Cohort']} / "
                f"Cases={row['Cases']}, "
                f"Controls={row['Controls']}")

In [ ]:
# ==========================================
# コホート × Tierごとに全モデルを比較
# ==========================================

model_order = list(modelDs.keys())

cohort = 'SexStratified'

for tier in ['tier1']:

        # 各モデルの結果を結合
    plotD = []

    for model in model_order:

        temp = resultD[(model, cohort)].copy()
        temp = temp[temp['Tier'] == tier].copy()

        temp['Model'] = model

        plotD.append(temp)

    plotDF = pd.concat(
            plotD,
            ignore_index=True)

    if len(plotDF) == 0:
        continue

        # 疾患の並び順
    disease_order = (
            plotDF
            .groupby('Variable')['OR']
            .mean()
            .sort_values()
            .index)

        # ==========================================
        # plot
        # ==========================================

    fig, ax = plt.subplots(
            figsize=(15, max(7, len(disease_order) * 1.0)))

    disease_spacing = 2.5
        
    y_positions = {disease: i * disease_spacing
                       for i, disease in enumerate(disease_order)}

        # モデルごとの位置
    offsets = np.linspace(
            -0.6,
            0.6,
            len(model_order))

    for offset, model in zip(offsets, model_order):

        temp = plotDF[plotDF['Model'] == model].copy()

        temp = (
                temp.set_index('Variable')
                .reindex(disease_order)
                .reset_index())

        for i, row in temp.iterrows():

            if pd.isna(row['OR']):
                continue

                # 有意なら濃く、非有意なら薄く
            if row['significant']:
                alpha = 1.0
            else:
                alpha = 0.3

            ax.errorbar(
                    row['OR'],
                    y_positions[row['Variable']] + offset,
                    xerr=[[
                        row['OR'] - row['CI_low']
                    ], [
                        row['CI_high'] - row['OR']
                    ]],
                    fmt='o',
                    color=model_colors[model],
                    ecolor=model_colors[model],
                    capsize=5,
                    elinewidth=2.5,
                    alpha=alpha,
                    label=model if i == 0 else None)

    ax.axvline(
            1,
            linestyle='--',
            linewidth=1)

    ax.set_yticks([y_positions[d] for d in disease_order])

    ax.set_yticklabels(disease_order, fontsize=28)

    ax.set_xlabel('ODDs Ratio (95% CI)', fontsize=28)

    legend_handles = [
            Line2D(
                [0], [0],
                color=model_colors[model],
                marker='o',
                linestyle='-',
                label=model)
            for model in model_order]

    ax.legend(
            handles=legend_handles,
            bbox_to_anchor=(1.02, 1),
            fontsize=24,
            loc='upper left',
            title='Model')

    plt.tight_layout()
    plt.show()

# end of notebook